# iRage AlgoArena — Short-Horizon Return Prediction
## Final Submission Notebook (v8)

**Pipeline Summary**

| Step | Detail |
|---|---|
| Data | 661,574 train rows · 410,139 test rows · 445 features |
| Feature Engineering | Momentum (Acc12/Acc23/Div13), AbsMom, Price feats, Family Agg, SO3_T poly — 471 new features |
| Feature Selection | Top 300 by gain (1-fold LGB pass, train-only) |
| Normalization | Z-score using train stats only |
| CV | GroupKFold(5) on CV_GROUP (70 groups) |
| Models | LGB Huber (num_leaves=15) · LGB Shallow (num_leaves=7) · CatBoost Huber |
| Ensemble | Nelder-Mead OOF weight optimization |
| Target | Clip P2/P98, Huber loss (kurtosis=48 fat tails) |
| Runtime | ~28 min on CPU (< 30 min limit) |

**Compliance**
- Data read from `/kaggle/input/competitions/short-horizon-return-prediction-challenge-by-i-rage/`
- No internet, no pip install, no external data
- No cross-sample test operations (all FE is row-wise)
- Normalization uses train statistics only
- GroupKFold on CV_GROUP (no temporal leakage)
- Generates `submission.csv` with 410,139 rows


In [ ]:
# CELL 1 - IMPORTS
import os, gc, warnings, time
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from pathlib import Path
from scipy.optimize import minimize

import lightgbm as lgb
from sklearn.model_selection import GroupKFold, KFold
from sklearn.metrics import r2_score

try:
    from catboost import CatBoostRegressor, Pool
    CATBOOST_AVAILABLE = True
    print('CatBoost OK')
except ImportError:
    CATBOOST_AVAILABLE = False
    print('CatBoost not available')

SEED = 23
np.random.seed(SEED)
NOTEBOOK_START = time.time()
print(f'LightGBM {lgb.__version__}')
print('Imports OK')

In [ ]:
# CELL 2 - CONFIG
#
# v7 builds on OLD v6 (best result: 0.001476).
# Core fix: reduce model capacity to force universal patterns.
#
# REGIME MEMORIZATION (why train/val = 150x):
#   With num_leaves=31 and 661k rows, each tree can partition data
#   into 31 very specific buckets. These buckets fit training regimes
#   perfectly but are meaningless in held-out regimes.
#   Solution: num_leaves=15 forces each split to cover 2x more data
#   -> only patterns universal enough to span large populations survive.
#
# HUBER LOSS JUSTIFICATION:
#   Target kurtosis=48. Under MSE, a return of 0.1 (3x std) contributes
#   0.01 to the loss vs 0.001 for a 0.01 return = 10x weight.
#   Huber caps contribution at delta: alpha=0.02 means returns > 0.02
#   are treated with L1 (linear) instead of L2 (quadratic) penalty.
#   This prevents the ~2% extreme returns from dominating gradients.
#
# CATBOOST HUBER:
#   CatBoost syntax: loss_function='Huber:delta=0.05'
#   delta=0.05: cap at 1.4x std of clipped target (std~0.036, clip P2/P98)
#   eval_metric stays RMSE (for early stopping on validation)

class CFG:
    DATA_DIR = Path('/kaggle/input/competitions/short-horizon-return-prediction-challenge-by-i-rage')
    TRAIN_PATH = DATA_DIR / 'train.parquet'
    TEST_PATH  = DATA_DIR / 'test.parquet'
    SAMPLE_SUB = DATA_DIR / 'sample_submission.csv'
    OUTPUT_DIR = Path('/kaggle/working')
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    N_FOLDS = 5
    SEED    = 23

    # P2/P98: confirmed better than P1/P99 from v5 vs v6 comparison
    TARGET_CLIP = True
    CLIP_LOWER  = 2
    CLIP_UPPER  = 98

    BUILD_MOMENTUM     = True
    BUILD_ABS_MOMENTUM = True
    BUILD_FAMILY_AGG   = True
    BUILD_PRICE_FEATS  = True
    BUILD_SO3_POLY     = True

    # Feature selection: keep top 300 by gain importance (OLD v6 approach)
    N_TOP_FEATURES     = 300

    # Z-score using train stats only (new v6 fix - kept)
    STANDARDIZE        = True

    # ── MODEL A: LGB with Huber loss ──────────────────────────
    # num_leaves=15: halved from 31 -> forces universal splits
    # min_child_samples=200: each leaf needs more evidence
    # lambda_l2=8: stronger shrinkage vs v6 old (3)
    # Huber alpha=0.02: near-MAE, robust to kurtosis=48
    LGB_HUBER_PARAMS = {
        'objective':         'huber',
        'alpha':             0.02,      # near-MAE; optimal for Laplace return distribution
        'metric':            'huber',
        'learning_rate':     0.02,
        'num_leaves':        15,        # v7: halved from 31
        'max_depth':         -1,
        'min_child_samples': 200,       # v7: doubled from 100
        'feature_fraction':  0.6,
        'bagging_fraction':  0.8,
        'bagging_freq':      1,
        'lambda_l1':         0.5,
        'lambda_l2':         8.0,       # v7: up from 3
        'min_gain_to_split': 0.0,       # fixed: was 0.01 in v5 (killed signal)
        'verbose':           -1,
        'n_jobs':            -1,
        'seed':              SEED,
    }

    # ── MODEL B: LGB Shallow (7 leaves = universal patterns only) ─
    # key insight from v7: LGB_Shallow ran 570 iter on 7-leaf trees
    # meaning genuinely universal patterns exist that survive large-leaf splits.
    # Kept. LGB_Q50 removed (0.000784 OOF, same cost as Huber, less value).
    LGB_SHALLOW_PARAMS = {
        'objective':         'huber',
        'alpha':             0.02,
        'metric':            'huber',
        'learning_rate':     0.05,      # faster: fewer useful patterns to find
        'num_leaves':        7,         # very shallow: global patterns only
        'max_depth':         -1,
        'min_child_samples': 500,       # very large: only rock-solid patterns
        'feature_fraction':  0.5,
        'bagging_fraction':  0.8,
        'bagging_freq':      1,
        'lambda_l1':         1.0,
        'lambda_l2':         15.0,      # heavily regularized
        'min_gain_to_split': 0.0,
        'verbose':           -1,
        'n_jobs':            -1,
        'seed':              SEED + 2,
    }

    # ── MODEL C: CatBoost with Huber loss ─────────────────────
    # v7 DIAGNOSIS: CAT ran only 137 iterations (best at ~37, waited 100).
    # depth=4 + min_data_in_leaf=200 made splits too restrictive.
    # Model converged in 37 iter because it ran out of valid patterns.
    #
    # v8 FIXES:
    # depth=5 (back from 4): 32 leaves = more expressiveness to learn from
    # min_data_in_leaf=150 (from 200): slightly less restrictive splits
    # od_wait=50 (from 100): stop sooner after plateau, but model finds more
    #   patterns before plateau with relaxed constraints -> more total iterations
    # l2_leaf_reg=10 (from 8): compensate depth=5 expressiveness with regularization
    # Expected: ~300-500 iterations (vs 137 in v7)
    CAT_PARAMS = dict(
        loss_function='Huber:delta=0.05',
        eval_metric='RMSE',
        iterations=1200,
        learning_rate=0.04,
        depth=5,                         # v8: back to 5 (was 4, caused under-training)
        l2_leaf_reg=10.0,                # v8: up from 8 to compensate depth=5
        min_data_in_leaf=150,            # v8: relaxed from 200 (too restrictive)
        border_count=128,
        random_seed=SEED,
        od_type='Iter',
        od_wait=50,                      # v8: tighter (50 vs 100) - stops faster after plateau
        verbose=200,
        task_type='CPU',
        thread_count=-1,
        allow_writing_files=False,
    )

    LGB_NUM_ROUNDS = 5000
    LGB_EARLY_STOP = 200

    RUNTIME_LIMIT_S   = 1620   # 27 min hard stop
    CAT_SECS_PER_ITER = 0.231  # from v4 timing

print('Config OK')

In [ ]:
# CELL 3 - DATA LOADING
print('Loading data...')
t0 = time.time()

train = pd.read_parquet(CFG.TRAIN_PATH)
test  = pd.read_parquet(CFG.TEST_PATH)

print(f'Train: {train.shape} | Test: {test.shape}')
print(f'Loaded in {time.time()-t0:.1f}s')

train_ids = train['ID'].values.copy()
test_ids  = test['ID'].values.copy()
y_raw     = train['TARGET'].copy()

if 'CV_GROUP' in train.columns:
    cv_groups    = train['CV_GROUP'].values.copy()
    USE_GROUP_CV = True
    n_groups     = len(np.unique(cv_groups))
    print(f'CV_GROUP: {n_groups} groups -> GroupKFold')
    train.drop(columns=['ID', 'TARGET', 'CV_GROUP'], inplace=True)
else:
    cv_groups    = None
    USE_GROUP_CV = False
    print('No CV_GROUP -> KFold fallback')
    train.drop(columns=['ID', 'TARGET'], inplace=True)

test.drop(columns=['ID'], inplace=True)

assert list(train.columns) == list(test.columns), 'FATAL: column mismatch!'
print(f'Features: {train.shape[1]} | Elapsed: {time.time()-NOTEBOOK_START:.0f}s')

In [ ]:
# CELL 4 - TARGET CLIPPING
# P2/P98: proven better than P1/P99 (v5 vs OLD v6 comparison).
# Removes extreme microstructure noise (kurtosis=48).
# TRAINING ONLY. Test predictions unconstrained.

desc = y_raw.describe(percentiles=[0.01, 0.02, 0.05, 0.25, 0.5, 0.75, 0.95, 0.98, 0.99])
print(desc)
print(f'Kurtosis: {y_raw.kurtosis():.2f} (fat tails - justifies Huber/MAE loss)')

clip_lo  = y_raw.quantile(CFG.CLIP_LOWER / 100)
clip_hi  = y_raw.quantile(CFG.CLIP_UPPER / 100)
y_series = y_raw.clip(clip_lo, clip_hi)
n_clip   = int(((y_raw < clip_lo) | (y_raw > clip_hi)).sum())
print(f'Clip P{CFG.CLIP_LOWER}/P{CFG.CLIP_UPPER}: [{clip_lo:.6f}, {clip_hi:.6f}]')
print(f'Clipped: {n_clip:,} rows ({100*n_clip/len(y_raw):.2f}%)')
print(f'Std: {y_raw.std():.6f} -> {y_series.std():.6f}')

y = y_series.values.astype(np.float32)
print('Target ready OK')

In [ ]:
# CELL 5 - FEATURE CATEGORIZATION
all_cols  = train.columns.tolist()
lag1_cols = [c for c in all_cols if c.endswith('_LagT1')]
lag2_cols = [c for c in all_cols if c.endswith('_LagT2')]
lag3_cols = [c for c in all_cols if c.endswith('_LagT3')]
lag_set   = set(lag1_cols + lag2_cols + lag3_cols)
base_cols = [c for c in all_cols if c not in lag_set]

special_cols   = ['Price', 'SO3_T']
pure_base_cols = [c for c in base_cols if c not in special_cols]

def get_family(col):
    return col.split('_')[0]

families = sorted(set(get_family(c) for c in pure_base_cols))

lag_map = {
    c: {
        'lag1': c+'_LagT1' if c+'_LagT1' in lag_set else None,
        'lag2': c+'_LagT2' if c+'_LagT2' in lag_set else None,
        'lag3': c+'_LagT3' if c+'_LagT3' in lag_set else None,
    } for c in pure_base_cols
}

print(f'Pure base: {len(pure_base_cols)} | Lag1/2/3: {len(lag1_cols)}/{len(lag2_cols)}/{len(lag3_cols)}')
print(f'Families: {families} | Total: {len(all_cols)}')

In [ ]:
# CELL 6 - FEATURE ENGINEERING
# ALL operations are purely row-wise. No cross-sample test operations.
# Same function applied identically to train and test.

def feature_engineering(df, lag_map, families, pure_base_cols, verbose=True):
    df = df.copy()
    new_feats = []

    # A. MOMENTUM: Acc12=LagT1-LagT2, Acc23=LagT2-LagT3, Div13=LagT1-LagT3
    if CFG.BUILD_MOMENTUM:
        n0 = len(new_feats)
        for bc, lags in lag_map.items():
            l1, l2, l3 = lags['lag1'], lags['lag2'], lags['lag3']
            if l1 and l2 and l1 in df.columns and l2 in df.columns:
                df[bc+'_Acc12'] = df[l1]-df[l2]; new_feats.append(bc+'_Acc12')
            if l2 and l3 and l2 in df.columns and l3 in df.columns:
                df[bc+'_Acc23'] = df[l2]-df[l3]; new_feats.append(bc+'_Acc23')
            if l1 and l3 and l1 in df.columns and l3 in df.columns:
                df[bc+'_Div13'] = df[l1]-df[l3]; new_feats.append(bc+'_Div13')
        if verbose: print(f'  Momentum: {len(new_feats)-n0}')

    # B. |LagT1|: momentum magnitude (trees cannot learn |x| from x alone)
    if CFG.BUILD_ABS_MOMENTUM:
        n0 = len(new_feats)
        for bc, lags in lag_map.items():
            l1 = lags['lag1']
            if l1 and l1 in df.columns:
                df[bc+'_AbsMom'] = df[l1].abs(); new_feats.append(bc+'_AbsMom')
        if verbose: print(f'  AbsMom:   {len(new_feats)-n0}')

    # C. Price-specific: Price_Acc, Price_Div13, Price_AbsMom
    if CFG.BUILD_PRICE_FEATS and 'Price' in df.columns:
        n0 = len(new_feats)
        pl1, pl2, pl3 = 'Price_LagT1', 'Price_LagT2', 'Price_LagT3'
        if pl1 in df.columns and pl2 in df.columns:
            df['Price_Acc']    = df[pl1]-df[pl2]; new_feats.append('Price_Acc')
        if pl1 in df.columns and pl3 in df.columns:
            df['Price_Div13']  = df[pl1]-df[pl3]; new_feats.append('Price_Div13')
        if pl2 in df.columns and pl3 in df.columns:
            df['Price_Acc23']  = df[pl2]-df[pl3]; new_feats.append('Price_Acc23')
        if pl1 in df.columns:
            df['Price_AbsMom'] = df[pl1].abs();   new_feats.append('Price_AbsMom')
        if verbose: print(f'  Price:    {len(new_feats)-n0}')

    # D. Family aggregates: cross-instrument consensus / dispersion signals
    if CFG.BUILD_FAMILY_AGG:
        n0 = len(new_feats)
        for fam in families:
            fb = [c for c in pure_base_cols if get_family(c)==fam and c in df.columns]
            fl1 = [c+'_LagT1' for c in fb if c+'_LagT1' in df.columns]
            if len(fb) < 2: continue
            df[fam+'_BaseMean'] = df[fb].mean(axis=1); new_feats.append(fam+'_BaseMean')
            df[fam+'_BaseStd']  = df[fb].std(axis=1);  new_feats.append(fam+'_BaseStd')
            if len(fl1) >= 2:
                fmom = df[fl1]
                df[fam+'_MomMean']   = fmom.mean(axis=1); new_feats.append(fam+'_MomMean')
                df[fam+'_MomStd']    = fmom.std(axis=1);  new_feats.append(fam+'_MomStd')
                df[fam+'_MomSpread'] = fmom.max(axis=1)-fmom.min(axis=1); new_feats.append(fam+'_MomSpread')
        if verbose: print(f'  FamilyAgg:{len(new_feats)-n0}')

    # E. SO3_T polynomial + interactions
    if CFG.BUILD_SO3_POLY and 'SO3_T' in df.columns:
        n0 = len(new_feats)
        df['SO3_T_sq'] = df['SO3_T']**2; new_feats.append('SO3_T_sq')
        df['SO3_T_cb'] = df['SO3_T']**3; new_feats.append('SO3_T_cb')
        if 'Price_LagT1' in df.columns:
            df['SO3T_x_PriceMom'] = df['SO3_T']*df['Price_LagT1']
            new_feats.append('SO3T_x_PriceMom')
        for fam in families:
            fn = fam+'_MomMean'
            if fn in df.columns:
                df['SO3T_x_'+fam+'_Mom'] = df['SO3_T']*df[fn]
                new_feats.append('SO3T_x_'+fam+'_Mom')
        if verbose: print(f'  SO3_T:    {len(new_feats)-n0}')

    if verbose: print(f'  Total new: {len(new_feats)} | Grand total: {df.shape[1]}')
    return df, new_feats


print('Running FE...')
t0 = time.time()
train_fe, new_feats = feature_engineering(train, lag_map, families, pure_base_cols)
test_fe,  _         = feature_engineering(test,  lag_map, families, pure_base_cols, verbose=False)
print(f'Done in {time.time()-t0:.1f}s | Train: {train_fe.shape} | Test: {test_fe.shape}')

In [ ]:
# CELL 7 - FINALIZE ARRAYS
train_fe.replace([np.inf, -np.inf], np.nan, inplace=True)
test_fe.replace( [np.inf, -np.inf], np.nan, inplace=True)
assert list(train_fe.columns) == list(test_fe.columns), 'FATAL: column mismatch!'

feature_cols = train_fe.columns.tolist()
n_features   = len(feature_cols)
print(f'Total features: {n_features}')

X      = train_fe.values.astype(np.float32)
X_test = test_fe.values.astype(np.float32)
print(f'X: {X.shape} | X_test: {X_test.shape}')

del train_fe, test_fe, train, test
gc.collect()
print(f'Memory freed OK | Elapsed: {time.time()-NOTEBOOK_START:.0f}s')

In [ ]:
# CELL 8 - FEATURE SELECTION (TOP 300 BY GAIN)
# Train one fast 500-round fold to compute gain importance.
# Keep top N_TOP_FEATURES (=300) by cumulative gain.
# TRAINING DATA ONLY: no test information used.
# Reduces noise features, speeds up full CV.

print(f'Feature selection: {len(feature_cols)} -> top {CFG.N_TOP_FEATURES} by gain...')
t_fs = time.time()

# Quick split for importance
if USE_GROUP_CV:
    _spl = list(GroupKFold(n_splits=CFG.N_FOLDS).split(X, y, groups=cv_groups))
else:
    _spl = list(KFold(n_splits=CFG.N_FOLDS, shuffle=True, random_state=CFG.SEED).split(X, y))

trn_fs, val_fs = _spl[0]

_fs_params = {**CFG.LGB_HUBER_PARAMS, 'learning_rate': 0.05, 'num_leaves': 31}
_d_trn = lgb.Dataset(X[trn_fs], label=y[trn_fs], feature_name=feature_cols, free_raw_data=True)
_d_val = lgb.Dataset(X[val_fs],  label=y[val_fs],  reference=_d_trn, free_raw_data=True)

_fs_model = lgb.train(
    _fs_params, _d_trn, num_boost_round=500,
    valid_sets=[_d_val], callbacks=[lgb.log_evaluation(-1)],
)

imp_gain = _fs_model.feature_importance(importance_type='gain')
imp_df   = pd.DataFrame({'feature': feature_cols, 'gain': imp_gain})
imp_df   = imp_df.sort_values('gain', ascending=False).reset_index(drop=True)
imp_df.to_csv(CFG.OUTPUT_DIR / 'feature_importance.csv', index=False)

n_zero    = (imp_df['gain'] == 0).sum()
n_keep    = min(CFG.N_TOP_FEATURES, len(feature_cols))
sel_feats = imp_df['feature'].iloc[:n_keep].tolist()
sel_idx   = [feature_cols.index(f) for f in sel_feats]

X          = X[:, sel_idx]
X_test     = X_test[:, sel_idx]
feature_cols = sel_feats
n_features   = len(feature_cols)

print(f'Zero-importance features: {n_zero}')
print(f'Selected: {n_features} features')
print(f'Top 5: {sel_feats[:5]}')
print(f'Done in {time.time()-t_fs:.1f}s')

del _fs_model, _d_trn, _d_val, imp_df
gc.collect()

In [ ]:
# CELL 9 - Z-SCORE STANDARDIZATION (TRAIN STATS ONLY)
# Removes inter-family scale mismatch. Train stats only -> no test leakage.
# Policy compliant: uses only simple aggregate statistics from training data.

if CFG.STANDARDIZE:
    t_std     = time.time()
    tr_mean   = np.nanmean(X, axis=0).astype(np.float32)
    tr_std    = np.nanstd(X,  axis=0).astype(np.float32)
    tr_std    = np.where(tr_std < 1e-8, 1.0, tr_std)

    X      = np.clip((X      - tr_mean) / tr_std, -10, 10).astype(np.float32)
    X_test = np.clip((X_test - tr_mean) / tr_std, -10, 10).astype(np.float32)
    X      = np.nan_to_num(X,      nan=0.0)
    X_test = np.nan_to_num(X_test, nan=0.0)

    print(f'Standardized | X mean={X.mean():.4f} std={X.std():.4f} | {time.time()-t_std:.1f}s')

print(f'Final arrays: X={X.shape} X_test={X_test.shape} y={y.shape}')
print(f'Elapsed: {time.time()-NOTEBOOK_START:.0f}s')

In [ ]:
# CELL 10 - CV SPLITTER
if USE_GROUP_CV:
    splitter = GroupKFold(n_splits=CFG.N_FOLDS)
    splits   = list(splitter.split(X, y, groups=cv_groups))
    print(f'GroupKFold on CV_GROUP ({n_groups} groups -> {CFG.N_FOLDS} folds)')
    for i, (ti, vi) in enumerate(splits):
        print(f'  Fold {i+1}: {len(ti):,} train | {len(vi):,} val | {len(np.unique(cv_groups[vi]))} groups')
else:
    splits = list(KFold(CFG.N_FOLDS, shuffle=True, random_state=CFG.SEED).split(X, y))
    print('KFold fallback')
print('CV ready OK')

In [ ]:
# CELL 11 - GENERIC LGB TRAINER
# Reusable function for all LGB variants.
# Returns: oof predictions, test predictions, list of best iterations, list of fold R2s

def run_lgb_cv(params, num_rounds, early_stop, splits, X, y, X_test,
               feature_cols, name='LGB', runtime_limit=CFG.RUNTIME_LIMIT_S):
    oof   = np.zeros(len(y), dtype=np.float64)
    ptest = np.zeros(len(X_test), dtype=np.float64)
    scores, iters = [], []

    print(f'--- {name} ---')
    for fold, (ti, vi) in enumerate(splits):
        elapsed = time.time() - NOTEBOOK_START
        if elapsed > runtime_limit * 0.9 and fold >= 3:
            print(f'  Time budget ({elapsed:.0f}s): stopping {name} at fold {fold}')
            break

        t0   = time.time()
        dtr  = lgb.Dataset(X[ti], label=y[ti], feature_name=feature_cols, free_raw_data=True)
        dval = lgb.Dataset(X[vi], label=y[vi], reference=dtr,             free_raw_data=True)

        m = lgb.train(
            params, dtr,
            num_boost_round=num_rounds,
            valid_sets=[dtr, dval],
            valid_names=['train','valid'],
            callbacks=[
                lgb.early_stopping(early_stop, verbose=False),
                lgb.log_evaluation(500),
            ],
        )
        bi = m.best_iteration
        iters.append(bi)

        vp        = m.predict(X[vi], num_iteration=bi)
        oof[vi]   = vp
        ptest    += m.predict(X_test, num_iteration=bi) / CFG.N_FOLDS

        # R2 on raw y (even if model trained on Huber/Quantile)
        fr2 = r2_score(y[vi], vp)
        scores.append(fr2)
        print(f'  Fold {fold+1}/{CFG.N_FOLDS} | R2: {fr2:+.6f} | iter: {bi} | {time.time()-t0:.0f}s')

        del m, dtr, dval
        gc.collect()

    n_done = len(scores)
    if n_done < CFG.N_FOLDS:
        ptest = ptest * CFG.N_FOLDS / n_done
        fi    = np.concatenate([splits[i][1] for i in range(n_done)])
        oof_r2 = r2_score(y[fi], oof[fi])
    else:
        oof_r2 = r2_score(y, oof)

    print(f'  {name} OOF R2: {oof_r2:+.6f} | std: {np.std(scores):.6f} | avg_iter: {np.mean(iters):.0f}')
    return oof, ptest, iters, scores, oof_r2

print('LGB trainer defined OK')

In [ ]:
# CELL 12 - RUN ALL LGB MODELS
print('=' * 55)
print(f'LightGBM CV - {CFG.N_FOLDS} folds x 2 models')
print('v8: LGB_Q50 removed (0.000784 OOF v7 = weakest, same cost)')
print('=' * 55)

oof_lgb_huber, pred_lgb_huber, iters_huber, scores_huber, r2_lgb_huber = run_lgb_cv(
    CFG.LGB_HUBER_PARAMS, CFG.LGB_NUM_ROUNDS, CFG.LGB_EARLY_STOP,
    splits, X, y, X_test, feature_cols, name='LGB_Huber'
)

oof_lgb_shal, pred_lgb_shal, iters_shal, scores_shal, r2_lgb_shal = run_lgb_cv(
    CFG.LGB_SHALLOW_PARAMS, CFG.LGB_NUM_ROUNDS, 100,
    splits, X, y, X_test, feature_cols, name='LGB_Shallow'
)

# Save for overfit check cell reference
lgb_iters = iters_huber
print(f'Elapsed: {time.time()-NOTEBOOK_START:.0f}s')

In [ ]:
# CELL 13 - CATBOOST WITH HUBER LOSS
# PRIMARY MODEL. Huber:delta=0.05 robust to kurtosis=48.
# depth=4 (vs 5): 16 leaves forces more universal splits.
# Dynamic time guard: reduce iterations if running low.

oof_cat    = np.zeros(len(y), dtype=np.float64)
pred_cat   = np.zeros(len(X_test), dtype=np.float64)
cat_scores, cat_iters = [], []
CATBOOST_DONE = False

if CATBOOST_AVAILABLE:
    print('=' * 55)
    print(f'CatBoost Huber - {CFG.N_FOLDS} folds (PRIMARY)')
    print('=' * 55)

    for fold, (ti, vi) in enumerate(splits):
        elapsed   = time.time() - NOTEBOOK_START
        remaining = CFG.RUNTIME_LIMIT_S - elapsed
        print(f'  Fold {fold+1} | {remaining:.0f}s remaining')

        if remaining < 60:
            print('  TIME EXHAUSTED - stopping CatBoost')
            break

        folds_left    = CFG.N_FOLDS - fold
        iter_budget   = int((remaining - 60) / folds_left / CFG.CAT_SECS_PER_ITER)
        iters_fold    = min(CFG.CAT_PARAMS['iterations'], max(400, iter_budget))
        if iters_fold < CFG.CAT_PARAMS['iterations']:
            print(f'  Reducing to {iters_fold} iter (time budget)')

        p_fold = {**CFG.CAT_PARAMS, 'iterations': iters_fold}
        t0     = time.time()
        pool_tr = Pool(X[ti], y[ti])
        pool_va = Pool(X[vi], y[vi])

        m = CatBoostRegressor(**p_fold)
        m.fit(pool_tr, eval_set=pool_va, use_best_model=True)

        vp         = m.predict(X[vi])
        oof_cat[vi] = vp
        pred_cat   += m.predict(X_test) / CFG.N_FOLDS

        fr2 = r2_score(y[vi], vp)
        bi  = m.get_best_iteration()
        cat_scores.append(fr2)
        cat_iters.append(bi)
        print(f'  Fold {fold+1}/{CFG.N_FOLDS} | R2: {fr2:+.6f} | best_iter: {bi} | {time.time()-t0:.0f}s')

        del m, pool_tr, pool_va
        gc.collect()

    n_done = len(cat_scores)
    if n_done == 0:
        print('No CatBoost folds completed — skipping safely')
        cat_r2 = 0.0
        pred_cat = np.zeros(len(X_test))
        CATBOOST_DONE = False
    elif n_done < CFG.N_FOLDS:
        pred_cat = pred_cat * CFG.N_FOLDS / n_done
        fi = np.concatenate([splits[i][1] for i in range(n_done)])
        cat_r2 = r2_score(y[fi], oof_cat[fi])
    else:
        cat_r2 = r2_score(y, oof_cat)

    cat_std       = np.std(cat_scores)
    CATBOOST_DONE = True
    print(f'CAT OOF R2: {cat_r2:+.6f} | std: {cat_std:.6f} | avg_iter: {np.mean(cat_iters):.0f}')
    print(f'Elapsed: {time.time()-NOTEBOOK_START:.0f}s')
else:
    cat_r2 = 0.0
    print('CatBoost not available')

In [ ]:
# CELL 14 - ENSEMBLE OPTIMIZATION
# OOF-based Nelder-Mead weight optimization.
# NO test-set information used. Softmax weights sum to 1.

model_names = ['LGB_Huber', 'LGB_Shallow']
oofs_list   = [oof_lgb_huber, oof_lgb_shal]
preds_list  = [pred_lgb_huber, pred_lgb_shal]

if CATBOOST_DONE:
    model_names.append('CAT_Huber')
    oofs_list.append(oof_cat)
    preds_list.append(pred_cat)

print('ENSEMBLE OPTIMIZATION')
print('-' * 50)
for nm, oof in zip(model_names, oofs_list):
    print(f'  {nm:<14} OOF R2: {r2_score(y, oof):+.6f}')

# Simple average
oof_sa  = np.mean(oofs_list, axis=0)
pred_sa = np.mean(preds_list, axis=0)
r2_sa   = r2_score(y, oof_sa)
print(f'Simple avg:  {r2_sa:+.6f}')

# Optimized weights
oofs_arr  = np.stack(oofs_list, axis=1)
preds_arr = np.stack(preds_list, axis=1)

def neg_r2(rw):
    w = np.exp(rw) / np.exp(rw).sum()
    return -r2_score(y, (oofs_arr * w).sum(axis=1))

best_r2_opt, best_w = -np.inf, None
for s in range(15):
    rng = np.random.RandomState(s)
    w0  = rng.randn(len(oofs_list)) * 0.1
    res = minimize(neg_r2, w0, method='Nelder-Mead',
                   options={'maxiter': 3000, 'xatol': 1e-7, 'fatol': 1e-9})
    if -res.fun > best_r2_opt:
        best_r2_opt = -res.fun
        best_w = np.exp(res.x) / np.exp(res.x).sum()

oof_opt  = (oofs_arr  * best_w).sum(axis=1)
pred_opt = (preds_arr * best_w).sum(axis=1)
r2_opt   = r2_score(y, oof_opt)

print(f'Optimized:   {r2_opt:+.6f}')
for nm, w in zip(model_names, best_w):
    print(f'  Weight {nm}: {w:.4f}')

if r2_opt > r2_sa:
    final_oof, final_pred = oof_opt, pred_opt
    final_r2, method_str  = r2_opt, 'Optimized wts'
else:
    final_oof, final_pred = oof_sa, pred_sa
    final_r2, method_str  = r2_sa, 'Simple avg'

print(f'FINAL: {final_r2:+.6f} ({method_str})')

In [ ]:
# CELL 15 - SUBMISSION GUARDS AND SAVE
CFG.OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

submission = pd.DataFrame({'ID': test_ids, 'TARGET': final_pred})

assert len(submission)            == len(test_ids),   'FATAL: row count'
assert list(submission.columns)  == ['ID','TARGET'],  'FATAL: columns'
assert submission['ID'].nunique() == len(test_ids),   'FATAL: dup IDs'
assert not submission['TARGET'].isnull().any(),        'FATAL: NaN'
assert not np.isinf(submission['TARGET'].values).any(),'FATAL: Inf'
assert submission['TARGET'].std() > 0,                 'FATAL: constant'

print('All 6 guards passed OK')

sub_path = CFG.OUTPUT_DIR / 'submission.csv'
submission.to_csv(sub_path, index=False)

v = pd.read_csv(sub_path)
assert len(v) == len(test_ids)
assert list(v.columns) == ['ID','TARGET']

total_elapsed = time.time() - NOTEBOOK_START
print(f'submission.csv SAVED | {len(submission):,} rows | {total_elapsed:.0f}s = {total_elapsed/60:.1f} min')
print(submission['TARGET'].describe())

In [ ]:
# CELL 16 - FINAL SCORECARD
total_elapsed = time.time() - NOTEBOOK_START

print('=' * 70)
print('  R2 SCORECARD - iRage Short-Horizon Return Prediction v8')
print('=' * 70)
print(f'  Features orig -> selected: {n_features + len(feature_cols) - len(feature_cols)} -> {n_features}')
print(f'  Train: {len(X):,} | Test: {len(X_test):,}')
print(f'  CV: {"GroupKFold(CV_GROUP)" if USE_GROUP_CV else "KFold(shuffle)"}')
print(f'  Runtime: {total_elapsed:.0f}s = {total_elapsed/60:.1f} min')
print('-' * 70)
print('  MODEL OOF R2')
for nm, oof in zip(model_names, oofs_list):
    r2  = r2_score(y, oof)
    bar = '#' * min(50, max(0, int(r2 * 1000)))
    print(f'  {nm:<14} {r2:+.6f}  {bar}')
print('-' * 70)
print('  ENSEMBLE')
print(f'  Simple avg:    {r2_sa:+.6f}')
print(f'  Optimized:     {r2_opt:+.6f}')
print(f'  FINAL ({method_str}): {final_r2:+.6f}')
print('-' * 70)
print('  OVERFIT DIAGNOSTIC')
print(f'  LGB_Huber avg best_iter: {np.mean(iters_huber):.0f}')
print(f'  LGB_Shallow avg best_iter: {np.mean(iters_shal):.0f}')
if CATBOOST_DONE: print(f'  CAT avg best_iter: {np.mean(cat_iters):.0f}')
print(f'  NOTE: Train R2 >> Val R2 is expected in this problem.')
print(f'  Focus on: Val R2 > 0 and consistent across folds.')
print('-' * 70)
print('  v8 CHANGES vs v7 (all evidence-based)')
print('  + Removed LGB_Q50 (0.000784 OOF v7 = weakest, saves ~9 min)')
print('  + CAT depth=5 (was 4: under-trained at 137 iter in v7)')
print('  + CAT min_data_in_leaf=150 (was 200: too restrictive)')
print('  + CAT od_wait=50 (was 100: faster response to plateau)')
print('  + CAT l2_leaf_reg=10 (compensates depth=5 expressiveness)')
print('  = Kept: LGB_Huber, LGB_Shallow, feature selection, z-score')
print('  v7 baseline: 0.001776 | Target: >0.002')
print('-' * 70)
print('  COMPLIANCE')
print('  OK Data: /kaggle/input/competitions/...')
print('  OK No internet / pip / external data')
print('  OK GroupKFold on CV_GROUP')
print('  OK Zero cross-sample test operations')
print('  OK Feature normalization uses train stats only')
runtime_ok = total_elapsed < 1800
print(f'  {"OK" if runtime_ok else "FAIL"} Runtime {total_elapsed/60:.1f} min {"< 30 min" if runtime_ok else "> 30 min"}')
print('=' * 70)
print()

if final_r2 > 0.005:  print('EXCEPTIONAL - top leaderboard')
elif final_r2 > 0.003:print('STRONG - prize range')
elif final_r2 > 0.002:print('GOOD - above majority')
elif final_r2 > 0.001:print('MODERATE - above baseline')
elif final_r2 > 0:    print('WEAK - positive signal')
else:                  print('NO SIGNAL')

print()
print('NOTEBOOK COMPLETE v8')
if np.mean(iters_huber) < 100:
    print('WARNING: LGB_Huber avg iter < 100. Params may still be too restrictive.')
if CATBOOST_DONE and np.mean(cat_iters) < 100:
    print('WARNING: CAT avg iter < 100. depth=5 fix may not have resolved under-training.')
elif CATBOOST_DONE and np.mean(cat_iters) > 100:
    print(f'CAT OK: {np.mean(cat_iters):.0f} avg iter (was 137 in v7 - improvement expected)')

---
## SUBMISSION CHECKLIST (v8)

- [ ] Runtime < 30 min (Cell 16)
- [ ] LGB_Huber avg best_iter > 50 (fixed params should now run longer)
- [ ] Final OOF R2 > 0.001476 (beat OLD v6)
- [ ] Fold std < 0.003 per model (stable across regimes)
- [ ] submission.csv: 410,139 rows, columns [ID, TARGET]
- [ ] No NaN, no Inf in TARGET

## Requirement Audit

| Rule | Status |
|---|---|
| `/kaggle/input/competitions/...` exact path | PASS |
| No internet / pip install | PASS |
| CPU only (task_type=CPU) | PASS |
| No cross-sample test ranking/KNN/cluster | PASS |
| GroupKFold on CV_GROUP | PASS |
| Feature selection/normalization: train stats only | PASS |
| 30 min runtime | PASS (dynamic guard active) |